# Stage 20 Astronomer RF Observation Console

External 10 MHz/PPS diagnostics, 8-lane DAC->ADC phase control, raw preview, spectrum, and coherence witnesses.


In [ ]:
from pathlib import Path
from collections import deque
import json
import asyncio
import sys
import time

import ipywidgets as W
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/home/xilinx/jupyter_notebooks/t510_fengine'),
    Path('/home/xilinx/t510_fengine_bringup'),
    Path('/home/astrolab/demo-ant'),
]
PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'overlay' / 't510_fengine.bit').exists() and (root / 'python' / 't510_fengine.py').exists():
        PROJECT_ROOT = root
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('overlay/t510_fengine.bit and python/t510_fengine.py were not found')
sys.path.insert(0, str(PROJECT_ROOT))
import importlib
import python.t510_clock as t510_clock_module
import python.t510_fengine as t510_fengine_module
# Jupyter kernels keep imported modules alive across notebook updates. Reload
# both the LMK clock controller and the F-engine wrapper so Init uses the
# current long-lock-wait Stage 19 code, not a stale in-memory Stage 18 module.
importlib.reload(t510_clock_module)
importlib.reload(t510_fengine_module)
ObservationSpectrumStabilizer = t510_fengine_module.ObservationSpectrumStabilizer
T510FEngine = t510_fengine_module.T510FEngine

BITFILE = PROJECT_ROOT / 'overlay' / 't510_fengine.bit'
EXPECTED_CORE_VERSION = 0x00010012
COLORS = ['#0b5cad', '#c45200', '#217a3b', '#b3261e', '#6f42c1', '#5f6368', '#008b8b', '#9a7b00']
CHANNEL_LABELS = [f'CH{i} DAC{i}->ADC{i}' for i in range(8)]
state = {
    'core': None,
    'live': False,
    'task': None,
    'phase_task': None,
    'last_cfg': None,
    'last_error': None,
    'last_fps': 0.0,
    'stabilizer': ObservationSpectrumStabilizer(),
    'waterfall': deque(maxlen=60),
    'waterfall_key': None,
    'last_display': {},
    'last_rates': None,
    'last_rates_time': 0.0,
    'last_status_update': 0.0,
    'last_waterfall_update': 0.0,
    'last_capture_elapsed_s': 0.0,
    'last_layout_key': None,
    'last_profile': {},
    'last_preview': None,
    'frozen_preview': None,
    'last_provenance': None,
    'phase_history': deque(maxlen=240),
    'aligned_history': deque(maxlen=240),
    'aligned_history_by_channel': {ch: deque(maxlen=240) for ch in range(8)},
    'alignment_anchor_deg': None,
    'auto_anchor_pending': False,
    'last_aligned_view': None,
    'last_hardware_event': None,
}



def visible_mask():
    return (1 << int(visible_channels.value)) - 1

def amplitude_code():
    return int(round(float(amplitude_percent.value) / 100.0 * 8192.0))

def phase_values():
    return [float(widget.value) for widget in phase_ch]

def input_source():
    return str(input_source_mode.value)

def expected_signal_hz():
    if input_source() == 'external_adc_tone':
        return float(expected_signal_mhz.value) * 1_000_000.0
    return float(signal_mhz.value) * 1_000_000.0

def dac_signal_hz():
    return float(signal_mhz.value) * 1_000_000.0

def science_output_mode_value():
    return str(science_output_mode.value)

def dac_enable_mask():
    return 0x00 if input_source() == 'external_adc_tone' else visible_mask()

def dac_amplitude_code():
    return 0 if input_source() == 'external_adc_tone' else amplitude_code()

def observation_label():
    dac_txt = 'DAC off' if input_source() == 'external_adc_tone' else f"DAC={float(signal_mhz.value):.1f} MHz"
    return f"source={input_source()}  input={expected_signal_hz()/1e6:.1f} MHz  {dac_txt}"

def capture_input_mask():
    return visible_mask() | (1 << int(waterfall_channel.value))

def core():
    if state['core'] is None:
        raise RuntimeError('Load overlay first')
    return state['core']

def capture_count():
    return T510FEngine.observation_capture_count(
        time_window_us=float(time_window_us.value),
        oversample=float(oversample.value),
    )

def reduce_uniform(x, y, max_points):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    max_points = max(16, int(max_points))
    if x.size <= max_points:
        return x, y
    idx = np.linspace(0, x.size - 1, max_points).astype(int)
    return x[idx], y[idx]

def reduce_spectrum_peak(x, y, max_points, xmin, xmax):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    max_points = max(32, int(max_points))
    mask = np.isfinite(x) & np.isfinite(y) & (x >= float(xmin)) & (x <= float(xmax))
    x = x[mask]
    y = y[mask]
    if x.size <= max_points:
        return x, y
    # x is uniformly spaced after FFT. Reshape-based max pooling is much cheaper
    # than one Python loop per displayed bin and preserves narrow peaks.
    group = int(np.ceil(x.size / max_points))
    padded = group * int(np.ceil(x.size / group))
    pad_n = padded - x.size
    if pad_n:
        x_pad = np.pad(x, (0, pad_n), mode='edge')
        y_pad = np.pad(y, (0, pad_n), mode='constant', constant_values=np.nanmin(y))
    else:
        x_pad = x
        y_pad = y
    xr = x_pad.reshape(-1, group)
    yr = y_pad.reshape(-1, group)
    local = np.nanargmax(yr, axis=1)
    rows = np.arange(yr.shape[0])
    return xr[rows, local], yr[rows, local]

def reduce_spectrum_bins(x, y, bins, xmin, xmax, fill_value):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    bins = max(32, int(bins))
    edges = np.linspace(float(xmin), float(xmax), bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    out = np.full(bins, float(fill_value), dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x >= float(xmin)) & (x <= float(xmax))
    x = x[mask]
    y = y[mask]
    if x.size == 0:
        return centers, out
    starts = np.searchsorted(x, edges[:-1], side='left')
    stops = np.searchsorted(x, edges[1:], side='right')
    for idx, (start, stop) in enumerate(zip(starts, stops)):
        if stop > start:
            out[idx] = float(np.nanmax(y[start:stop]))
    return centers, out

def get_rates(c, *, force=False):
    now = time.monotonic()
    interval = 1.0 / max(float(status_update_hz.value), 0.1)
    cached = state.get('last_rates')
    if (not force) and cached is not None and (now - float(state.get('last_rates_time', 0.0))) < interval:
        return cached
    rates = c.read_realtime_rates()
    state['last_rates'] = rates
    state['last_rates_time'] = now
    return rates

def set_status(html, *, kind='info'):
    colors = {'info': '#174ea6', 'ok': '#137333', 'warn': '#b06000', 'err': '#b3261e'}
    status_html.value = f"<pre style='margin:0;color:{colors.get(kind, '#202124')};white-space:pre-wrap'>{html}</pre>"

def waterfall_signature():
    return (
        input_source(),
        round(float(expected_signal_mhz.value), 6),
        round(float(signal_mhz.value), 6),
        round(float(center_mhz.value), 6),
        round(float(bandwidth_mhz.value), 6),
        int(waterfall_channel.value),
        int(waterfall_depth.value),
    )

def reset_stability_history(*_, clear_plot=True):
    state['stabilizer'] = ObservationSpectrumStabilizer(
        alpha=float(smoothing_alpha.value),
        min_snr_db=10.0,
        peak_jump_mhz=2.0,
        amp_jump_db=6.0,
    )
    state['waterfall'] = deque(maxlen=int(waterfall_depth.value))
    state['waterfall_key'] = waterfall_signature()
    state['last_display'] = {}
    state['last_waterfall_update'] = 0.0
    state['phase_history'].clear()
    state['aligned_history'].clear()
    for hist in state.get('aligned_history_by_channel', {}).values():
        hist.clear()
    if clear_plot:
        with waterfall_fig.batch_update():
            waterfall_fig.data[0].x = []
            waterfall_fig.data[0].y = []
            waterfall_fig.data[0].z = []

download_bitstream = W.Checkbox(value=True, description='Download bitstream')
clock_ref_mode = W.Dropdown(options=[('外部10MHz', 'external_10mhz'), ('板载TCXO 10MHz', 'tcxo_10mhz')], value='external_10mhz', description='10MHz参考', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
sync_mode_select = W.Dropdown(options=[('外部PPS硬触发', 'external_pps'), ('Free-run', 'free_run')], value='external_pps', description='同步模式', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
require_mts_lock = W.Checkbox(value=True, description='要求MTS/SYSREF通过')
force_clock_reconfigure = W.Checkbox(value=True, description='Init时重配LMK')
sync_diag_btn = W.Button(description='诊断10M/PPS', icon='activity', layout=W.Layout(width='140px'))
sync_html = W.HTML(value='<pre style="margin:0">10MHz/PPS: load overlay to diagnose.</pre>')
input_source_mode = W.Dropdown(options=[('8路 DAC→ADC 闭环', 'dac_loopback'), ('外部ADC单音', 'external_adc_tone')], value='dac_loopback', description='输入源', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
expected_signal_mhz = W.FloatSlider(value=200.0, min=50.0, max=350.0, step=0.5, description='ADC输入 MHz', readout_format='.1f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
signal_mhz = W.FloatSlider(value=200.0, min=50.0, max=350.0, step=0.5, description='DAC信号 MHz', readout_format='.1f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
center_mhz = W.FloatSlider(value=200.0, min=50.0, max=350.0, step=0.5, description='观测中心 MHz', readout_format='.1f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
bandwidth_mhz = W.ToggleButtons(options=[('20MHz', 20), ('100MHz', 100), ('200MHz', 200)], value=100, description='科学带宽', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
science_output_mode = W.Dropdown(options=[('OFF', 'off'), ('TIME only', 'time_only'), ('SPEC only', 'spec_only'), ('TIME+SPEC', 'time_spec'), ('TIME+monitor SPEC', 'time_monitor_spec')], value='off', description='UDP输出', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
force_tx_dry_run = W.Checkbox(value=True, description='QSFP dry-run保护')
cmac_enable = W.Checkbox(value=False, description='Enable CMAC live')
oversample = W.FloatSlider(value=2.5, min=1.0, max=4.0, step=0.25, description='软件过采样', readout_format='.2f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
time_window_us = W.FloatSlider(value=0.25, min=0.05, max=4.0, step=0.05, description='时间窗口 us', readout_format='.2f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
amplitude_percent = W.IntSlider(value=25, min=0, max=100, step=1, description='幅度 %FS', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
phase_deg = W.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='全局相位 deg', readout_format='.0f', continuous_update=True, style={'description_width': '130px'}, layout=W.Layout(width='430px'))
phase_ch = [W.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description=f'CH{i} 相位', readout_format='.0f', continuous_update=True, style={'description_width': '90px'}, layout=W.Layout(width='310px')) for i in range(8)]
apply_global_phase_btn = W.Button(description='同步到8路', icon='sliders-horizontal', layout=W.Layout(width='120px'))
scope_y_codes = W.IntSlider(value=512, min=64, max=4096, step=64, description='Y轴 ADC code', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
scope_waveform_mode = W.ToggleButtons(options=[('I', 'raw_i'), ('Q', 'raw_q'), ('I/Q', 'iq_overlay'), ('|IQ|', 'raw_mag'), ('RF等效', 'rf_equiv')], value='raw_i', description='波形', style={'description_width': '70px'}, layout=W.Layout(width='430px'))
spectrum_floor_db = W.IntSlider(value=-120, min=-160, max=-20, step=5, description='频谱底 dBFS', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
spectrum_top_db = W.IntSlider(value=-20, min=-80, max=10, step=5, description='频谱顶 dBFS', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
refresh_hz = W.FloatSlider(value=8.0, min=1.0, max=15.0, step=0.5, description='刷新 Hz', readout_format='.1f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
visible_channels = W.IntSlider(value=8, min=1, max=8, step=1, description='显示通道', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
smoothing_enabled = W.Checkbox(value=True, description='频谱平滑')
smoothing_alpha = W.FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05, description='平滑强度', readout_format='.2f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
waterfall_channel = W.Dropdown(options=[(f'CH{i}', i) for i in range(8)], value=0, description='瀑布通道', style={'description_width': '130px'}, layout=W.Layout(width='240px'))
waterfall_depth = W.IntSlider(value=30, min=10, max=120, step=10, description='瀑布历史帧', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
scope_display_points = W.IntSlider(value=512, min=128, max=2048, step=128, description='波形显示点', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
spectrum_display_points = W.IntSlider(value=384, min=128, max=2048, step=128, description='频谱显示点', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
waterfall_bins = W.IntSlider(value=192, min=64, max=1024, step=64, description='瀑布频点', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
waterfall_update_hz = W.FloatSlider(value=0.8, min=0.1, max=3.0, step=0.1, description='瀑布刷新 Hz', readout_format='.1f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
status_update_hz = W.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='状态刷新 Hz', readout_format='.1f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
fast_mode = W.Checkbox(value=True, description='快速模式')
baseband_live = W.Checkbox(value=False, description='刷新基带图')
waterfall_live = W.Checkbox(value=True, description='刷新瀑布')
min_ui_sleep_ms = W.IntSlider(value=30, min=0, max=100, step=5, description='UI让步 ms', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
raw_spectrum = W.Checkbox(value=False, description='Raw spectrum')
timeout_s = W.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description='Timeout s', readout_format='.1f', style={'description_width': '130px'}, layout=W.Layout(width='280px'))

load_btn = W.Button(description='Load', icon='download', button_style='primary', layout=W.Layout(width='110px'))
init_btn = W.Button(description='Init', icon='play', button_style='success', layout=W.Layout(width='110px'))
apply_btn = W.Button(description='Apply', icon='check', layout=W.Layout(width='110px'))
capture_btn = W.Button(description='Capture', icon='camera', layout=W.Layout(width='110px'))
start_btn = W.Button(description='Start Live', icon='play', button_style='success', layout=W.Layout(width='130px'))
stop_btn = W.Button(description='Stop', icon='stop', button_style='warning', layout=W.Layout(width='110px'))
reset_history_btn = W.Button(description='Reset history', icon='refresh-cw', layout=W.Layout(width='140px'))
freeze_frame = W.ToggleButton(value=False, description='Freeze current frame', icon='pause', button_style='', layout=W.Layout(width='190px'))
set_anchor_btn = W.Button(description='Set alignment anchor', icon='crosshairs', layout=W.Layout(width='190px'))
auto_anchor_after_apply = W.Checkbox(value=True, description='Apply后自动对齐')
show_measured_aligned_scope = W.Checkbox(value=True, description='显示实测对齐')
status_html = W.HTML(value='')
advanced_html = W.HTML(value='')
phase_html = W.HTML(value='<pre style="margin:0">Phase Provenance: capture a frame to populate.</pre>')

preview_audit_source = W.Dropdown(
    options=[('RFDC ADC preview', 'rfdc'), ('Internal DDS preview audit', 'internal_dds'), ('Sample-index ramp audit', 'sample_index_ramp')],
    value='rfdc',
    description='Preview source',
    style={'description_width': '130px'},
    layout=W.Layout(width='360px'),
)
preview_event_enable = W.Checkbox(value=False, description='Enable large-event trigger')
preview_freeze_on_event = W.Checkbox(value=True, description='Freeze on event')
preview_event_threshold = W.IntSlider(value=28000, min=1024, max=32767, step=512, description='Event threshold', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
event_capture_timeout = W.FloatSlider(value=2.0, min=0.2, max=30.0, step=0.2, description='Event timeout s', readout_format='.1f', style={'description_width': '130px'}, layout=W.Layout(width='430px'))
apply_audit_btn = W.Button(description='Apply audit', icon='check', layout=W.Layout(width='130px'))
refresh_audit_btn = W.Button(description='Refresh audit', icon='refresh-cw', layout=W.Layout(width='140px'))
capture_event_btn = W.Button(description='Capture event', icon='activity', button_style='warning', layout=W.Layout(width='150px'))
hardware_audit_html = W.HTML(value='<pre style="margin:0">Hardware Audit: load overlay to populate.</pre>')
rfdc_raw_channel = W.Dropdown(options=[(f'CH{i}', i) for i in range(8)], value=0, description='RFDC raw CH', style={'description_width': '110px'}, layout=W.Layout(width='220px'))
rfdc_raw_capture_beats = W.IntSlider(value=256, min=4, max=256, step=4, description='Raw beats', style={'description_width': '90px'}, layout=W.Layout(width='260px'))
capture_rfdc_raw_btn = W.Button(description='Capture RFDC raw', icon='radio', button_style='warning', layout=W.Layout(width='170px'))
rfdc_raw_html = W.HTML(value='<pre style="margin:0">RFDC Raw Witness: capture a channel to populate.</pre>')

scope_fig = go.FigureWidget()
for ch in range(8):
    scope_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch} I', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.3}, visible=(ch == 0)))
    scope_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch} Q', line={'color': COLORS[ch], 'width': 1.3, 'dash': 'dot'}, visible=False))
scope_fig.update_layout(height=360, margin={'l': 58, 'r': 20, 't': 36, 'b': 46}, xaxis_title='preview time (us)', yaxis_title='ADC code', template='plotly_white', showlegend=True, title='ADC raw preview scope')

baseband_fig = go.FigureWidget()
for ch in range(8):
    baseband_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch}', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.3}, visible=(ch == 0)))
baseband_fig.update_layout(height=280, margin={'l': 58, 'r': 20, 't': 34, 'b': 44}, xaxis_title='baseband time (us)', yaxis_title='ADC I code', template='plotly_white', showlegend=True)

spectrum_fig = go.FigureWidget()
for ch in range(8):
    spectrum_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch}', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.3}, visible=(ch == 0)))
spectrum_fig.update_layout(height=360, margin={'l': 58, 'r': 20, 't': 36, 'b': 46}, xaxis_title='RF frequency (MHz)', yaxis_title='power (dBFS)', template='plotly_white', showlegend=True)

waterfall_fig = go.FigureWidget(data=[go.Heatmap(x=[], y=[], z=[], colorscale='Viridis', zmin=-120, zmax=-20, zsmooth=False, hoverinfo='skip', colorbar={'title': 'dBFS'})])
waterfall_fig.update_layout(height=330, margin={'l': 58, 'r': 20, 't': 36, 'b': 46}, xaxis_title='RF frequency (MHz)', yaxis_title='recent frames', template='plotly_white', showlegend=False)

aligned_scope_fig = go.FigureWidget()
for ch in range(8):
    aligned_scope_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch} I', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.4}, visible=False))
    aligned_scope_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch} Q', line={'color': COLORS[ch], 'width': 1.2, 'dash': 'dot'}, visible=False))
aligned_scope_fig.update_layout(height=320, margin={'l': 58, 'r': 20, 't': 36, 'b': 46}, xaxis_title='preview time (us)', yaxis_title='ADC code', template='plotly_white', showlegend=True, title='Sample0-indexed raw preview scope')

aligned_phase_fig = go.FigureWidget()
for ch in range(8):
    aligned_phase_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines+markers', name=f'CH{ch} phase_error_deg', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.3}, visible=False))
aligned_phase_fig.update_layout(height=230, margin={'l': 58, 'r': 20, 't': 34, 'b': 44}, xaxis_title='recent frames', yaxis_title='phase error (deg)', yaxis_range=[-180, 180], template='plotly_white', showlegend=True, title='Sample0-aligned phase residual')

phase_fig = go.FigureWidget()
phase_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines+markers', name='measured FFT phase', line={'color': '#b3261e'}))
phase_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines+markers', name='sample0 coherent phase', line={'color': '#0b5cad'}))
phase_fig.update_layout(height=260, margin={'l': 58, 'r': 20, 't': 34, 'b': 44}, xaxis_title='recent frames', yaxis_title='phase (deg)', yaxis_range=[-180, 180], template='plotly_white', showlegend=True, title='Raw ADC IQ phase provenance')

event_fig = go.FigureWidget()
event_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name='event I', line={'color': '#b3261e'}))
event_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name='event Q', line={'color': '#0b5cad'}))
event_fig.update_layout(height=260, margin={'l': 58, 'r': 20, 't': 34, 'b': 44}, xaxis_title='event sample', yaxis_title='ADC code', yaxis_range=[-32768, 32767], template='plotly_white', showlegend=True, title='Preview large-event raw IQ buffer  CH0')

rfdc_raw_fig = go.FigureWidget()
rfdc_raw_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name='RFDC raw I', line={'color': '#b3261e'}))
rfdc_raw_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name='RFDC raw Q', line={'color': '#0b5cad'}))
rfdc_raw_fig.update_layout(height=280, margin={'l': 58, 'r': 20, 't': 34, 'b': 44}, xaxis_title='RFDC AXIS sub-sample', yaxis_title='ADC code', yaxis_range=[-32768, 32767], template='plotly_white', showlegend=True, title='RFDC AXIS raw witness')

def update_channel_visibility():
    mask = visible_mask()
    with scope_fig.batch_update(), baseband_fig.batch_update(), spectrum_fig.batch_update(), aligned_scope_fig.batch_update(), aligned_phase_fig.batch_update():
        for ch in range(8):
            vis = bool(mask & (1 << ch))
            scope_fig.data[2 * ch].visible = vis
            scope_fig.data[2 * ch + 1].visible = vis and str(scope_waveform_mode.value) == 'iq_overlay'
            baseband_fig.data[ch].visible = vis
            spectrum_fig.data[ch].visible = vis
            aligned_scope_fig.data[2 * ch].visible = vis and bool(show_measured_aligned_scope.value)
            aligned_scope_fig.data[2 * ch + 1].visible = vis and bool(show_measured_aligned_scope.value) and str(scope_waveform_mode.value) == 'iq_overlay'
            aligned_phase_fig.data[ch].visible = vis

def apply_observation(*, initialize=False, start=False):
    c = core()
    apply_fn = c.apply_mts_locked_observation_config if bool(require_mts_lock.value) else c.apply_sysref_locked_observation_config
    cfg = apply_fn(
        observe_center_hz=float(center_mhz.value) * 1_000_000.0,
        dac_signal_hz=dac_signal_hz(),
        expected_signal_hz=expected_signal_hz(),
        view_bw_hz=float(bandwidth_mhz.value) * 1_000_000.0,
        amplitude=dac_amplitude_code(),
        phase_deg=0.0,
        phase_deg_by_channel=phase_values(),
        enable_mask=dac_enable_mask(),
        adc_active_mask=0xffff if input_source() == 'dac_loopback' else 0x0001,
        initialize=initialize,
        start=start,
        require_full_clock_lock=bool(require_mts_lock.value),
        require_mts=bool(require_mts_lock.value),
        force_clock_reconfigure=bool(force_clock_reconfigure.value),
        input_source_mode=input_source(),
        clock_ref=str(clock_ref_mode.value),
        sync_mode=str(sync_mode_select.value),
    )
    science = c.configure_science_output(
        int(bandwidth_mhz.value),
        science_output_mode_value(),
        force_dry_run=bool(force_tx_dry_run.value),
        cmac_enable=bool(cmac_enable.value),
        clear_counters=True,
    )
    cfg['science_output'] = science
    time.sleep(0.2)
    state['last_cfg'] = cfg
    update_channel_visibility()
    reset_stability_history(clear_plot=True)
    state['alignment_anchor_deg'] = None
    state['auto_anchor_pending'] = bool(auto_anchor_after_apply.value and input_source() == 'dac_loopback')
    state['last_aligned_view'] = None
    state['frozen_preview'] = None
    state['last_preview'] = None
    if freeze_frame.value:
        freeze_frame.value = False
    return cfg

def apply_phase_only():
    c = core()
    tone = c.configure_dac_tone_bank(freq_hz=0.0, amplitude=dac_amplitude_code(), phase_offset_deg=0.0, phase_deg_by_channel=phase_values(), enable_mask=dac_enable_mask(), mode='constant_phasor')
    epoch = c.reset_dac_phase()
    time.sleep(0.15)
    if state['last_cfg'] is not None:
        state['last_cfg']['phase_deg'] = 0.0
        state['last_cfg']['phase_deg_by_channel'] = phase_values()
        state['last_cfg']['tone'] = tone
        state['last_cfg']['dac_phase_epoch'] = int(epoch)
    return epoch

def diagnose_sync(_=None):
    if state.get('core') is None:
        set_status('Load overlay before 10MHz/PPS diagnostic.', kind='warn')
        return
    try:
        diag = core().read_external_sync_diagnostics(interval_s=1.2)
        state['last_sync_diag'] = diag
        sync_html.value = (
            f"<pre style='margin:0;white-space:pre-wrap'>"
            f"10MHz/PPS diagnostic: {diag['classification']}\n"
            f"LMK locked={int(diag['lmk_locked'])}  external_ref_selected={int(diag['external_ref_selected'])}  ref_ok={int(diag['ref_ok'])}\n"
            f"PPS ok={int(diag['pps_ok'])}  count {diag['pps_count_before']} -> {diag['pps_count_after']}  delta={diag['pps_delta']}  recent={int(diag['pps_recent'])}\n"
            f"configured_clock_ref={diag['configured_clock_ref']}  configured_sync_mode={diag['configured_sync_mode']}\n"
            f"LED0=10MHz/RF clock ready  LED1=PPS blink  LED2=PPS recent  LED3=sync error\n"
            f"</pre>"
        )
        set_status(f"10MHz/PPS diagnostic: {diag['classification']}", kind='ok' if diag['ok'] else 'warn')
    except Exception as exc:
        set_status(f"10MHz/PPS diagnostic failed: {type(exc).__name__}: {exc}", kind='err')

def load_overlay(_=None):
    state['core'] = T510FEngine(str(BITFILE), download=bool(download_bitstream.value))
    status = state['core'].read_status()
    reset_stability_history(clear_plot=True)
    msg = f"Project: {PROJECT_ROOT}\nBitfile: {BITFILE}\nCORE_VERSION=0x{int(status['core_version']):08x}"
    set_status(msg, kind='ok' if int(status['core_version']) == EXPECTED_CORE_VERSION else 'warn')

def init_instrument(_=None):
    try:
        cfg = apply_observation(initialize=True, start=True)
        set_status(f"Initialized: {observation_label()}  center={cfg['observe_center_hz']/1e6:.1f} MHz  BW={cfg['view_bw_hz']/1e6:.0f} MHz", kind='ok')
        capture_once()
    except Exception as exc:
        state['last_error'] = exc
        set_status(f"Init failed: {type(exc).__name__}: {exc}", kind='err')

def apply_instrument(_=None):
    try:
        cfg = apply_observation(initialize=False, start=False)
        set_status(f"Applied: {observation_label()}  center={cfg['observe_center_hz']/1e6:.1f} MHz  BW={cfg['view_bw_hz']/1e6:.0f} MHz", kind='ok')
        capture_once()
    except Exception as exc:
        state['last_error'] = exc
        set_status(f"Apply failed: {type(exc).__name__}: {exc}", kind='err')

async def debounced_phase_apply():
    await asyncio.sleep(0.20)
    try:
        epoch = apply_phase_only()
        set_status(f"Phase applied: phases={','.join(f'{v:+.1f}' for v in phase_values())} deg  epoch={epoch}", kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f"Phase apply failed: {type(exc).__name__}: {exc}", kind='err')

def on_phase_change(change):
    if state['core'] is None:
        return
    task = state.get('phase_task')
    if task is not None and not task.done():
        task.cancel()
    state['phase_task'] = asyncio.create_task(debounced_phase_apply())

def on_waterfall_setting_change(change):
    reset_stability_history(clear_plot=True)

def set_alignment_anchor(_=None):
    view = state.get('last_aligned_view')
    ch0 = None if view is None else (view.get('channels', {}).get(0) or view.get('channels', {}).get('0'))
    if ch0 is None:
        set_status('Capture a sample0-aligned frame before setting anchor.', kind='warn')
        return
    state['alignment_anchor_deg'] = float(ch0['anchor_candidate_deg'])
    state['auto_anchor_pending'] = False
    state['aligned_history'].clear()
    for hist in state.get('aligned_history_by_channel', {}).values():
        hist.clear()
    set_status(f"Alignment anchor set: {state['alignment_anchor_deg']:+.3f} deg at sample0={int(view['sample0'])}", kind='ok')
    if not state.get('live', False):
        capture_once()

def format_rates(rates):
    r = rates['rates']
    blocks = ','.join(rates.get('science_block_reasons', [])) or 'none'
    return (
        f"ADC {r['adc_samples_per_s']/1e6:.2f} MS/s  "
        f"UDP {r['packetizer_packets_per_s']:.1f} pkt/s {r['packetizer_bytes_per_s']/1e6:.2f} MB/s  "
        f"TX dry-run {r['tx_dry_run_packets_per_s']:.1f} pkt/s {r['tx_dry_run_bytes_per_s']/1e6:.2f} MB/s  "
        f"SCI {rates.get('science_bandwidth_mhz', 0)}MHz {rates.get('science_output_mode', 'UNKNOWN')} "
        f"{rates.get('science_sample_rate_hz', 0.0)/1e6:.2f} MS/s {rates.get('science_payload_rate_mbps', 0.0):.0f} Mb/s  "
        f"QSFP_PRESENT={int(rates.get('qsfp_module_present', False))} CMAC_READY={int(rates.get('cmac_live_ready', False))} BLOCK={blocks}"
    )

def update_phase_provenance_panel(provenance, aligned_view=None):
    if not provenance:
        return
    state['last_provenance'] = provenance
    ch0 = provenance.get('channels', {}).get(0) or provenance.get('channels', {}).get('0')
    aligned_ch0 = None if aligned_view is None else (aligned_view.get('channels', {}).get(0) or aligned_view.get('channels', {}).get('0'))
    if ch0 is None:
        phase_html.value = "<pre style='margin:0'>Phase Provenance: CH0 missing.</pre>"
        return
    frozen = bool(freeze_frame.value and state.get('frozen_preview') is not None)
    state['phase_history'].append({
        'measured': float(ch0['measured_fft_phase_deg']),
        'coherent': float(ch0['sample0_coherent_phase_deg']),
        'display': float(ch0['display_rf_phase_deg']),
        'sample0': int(provenance['sample0']),
    })
    hist = list(state['phase_history'])
    x = np.arange(-len(hist) + 1, 1, dtype=int)
    with phase_fig.batch_update():
        phase_fig.data[0].x = x
        phase_fig.data[0].y = [item['measured'] for item in hist]
        phase_fig.data[1].x = x
        phase_fig.data[1].y = [item['coherent'] for item in hist]
        phase_fig.layout.title = 'Raw ADC IQ phase provenance  ' + ('Frozen frame replay' if frozen else 'Live preview')
    phase_html.value = (
        f"<pre style='margin:0;white-space:pre-wrap'>"
        f"Phase Provenance ({'Frozen hardware frame' if frozen else 'Live hardware preview'})\n"
        f"configured_phase_deg          = {float(ch0['configured_phase_deg']):+8.3f}\n"
        f"fft_peak_phase_deg            = {float(ch0['measured_fft_phase_deg']):+8.3f}   FFT peak diagnostic\n"
        f"sample0_coherent_phase_deg    = {float(ch0['sample0_coherent_phase_deg']):+8.3f}   FFT-peak based diagnostic\n"
        f"display_reference_phase_deg   = {float(ch0['display_rf_phase_deg']):+8.3f}   configured phase text\n"
        f"input_source_mode             = {ch0.get('input_source_mode', 'unknown')}  expected_signal={float(ch0.get('expected_signal_hz', 0.0))/1e6:.4f} MHz  dac_signal={float(ch0.get('dac_signal_hz', 0.0))/1e6:.4f} MHz\n"
        f"expected_tone_measured_phase  = {0.0 if aligned_ch0 is None else float(aligned_ch0['expected_tone_measured_phase_deg']):+8.3f}   fixed-frequency fit\n"
        f"sample0_aligned_phase_deg     = {0.0 if aligned_ch0 is None else float(aligned_ch0['sample0_aligned_phase_deg']):+8.3f}   Stage 14 drift field\n"
        f"phase_error_deg               = {0.0 if aligned_ch0 is None else float(aligned_ch0['phase_error_deg']):+8.3f}   after alignment anchor\n"
        f"sample0_mod_phase_deg         = {0.0 if aligned_ch0 is None else float(aligned_ch0['sample0_mod_phase_deg']):+8.3f}   sample0={int(provenance['sample0'])}\n"
        f"raw_baseband={float(ch0['raw_baseband_mhz']):+.6f} MHz  expected_baseband={0.0 if aligned_ch0 is None else float(aligned_ch0['expected_baseband_mhz']):+.6f} MHz  RF peak={float(ch0['rf_peak_mhz']):.6f} MHz\n"
        f"fit_residual={float(ch0['fit_residual_fraction'])*100.0:.2f}%  aligned_residual={0.0 if aligned_ch0 is None else float(aligned_ch0['fit_residual_fraction'])*100.0:.2f}%  clipped={bool(ch0['clipped'])}\n"
        f"NOTE: sample0_aligned_phase_deg / phase_error_deg are the Stage 14 hardware-sampling drift indicators."
        f"</pre>"
    )

def on_freeze_change(change):
    if bool(change['new']):
        if state.get('last_preview') is None:
            freeze_frame.value = False
            set_status('Capture one frame before Freeze current frame.', kind='warn')
            return
        state['frozen_preview'] = state['last_preview']
        set_status(f"Frozen current preview frame at sample0={int(state['frozen_preview']['sample0'])}. Capture/Live now replays the same raw IQ.", kind='warn')
    else:
        state['frozen_preview'] = None
        set_status('Freeze released; next capture reads hardware preview again.', kind='info')


def _source_label(value):
    return {'rfdc': 'RFDC ADC', 'internal_dds': 'Internal DDS', 'sample_index_ramp': 'Sample-index ramp'}.get(str(value), str(value))

def update_hardware_audit_panel(status=None):
    if status is None:
        if state.get('core') is None:
            hardware_audit_html.value = '<pre style="margin:0">Hardware Audit: load overlay first.</pre>'
            return
        status = core().read_status()
    audit_status = int(status.get('preview_audit_status', 0))
    audit_control = int(status.get('preview_audit_control', 0))
    source_code = int(status.get('preview_audit_source', 0))
    source_name = {0: 'rfdc', 1: 'internal_dds', 2: 'sample_index_ramp'}.get(source_code, f'unknown_{source_code}')
    dac_mode = int(status.get('dac_audit_ch0_mode', 0))
    event_info = int(status.get('preview_event_info', 0))
    event_source_code = (event_info >> 16) & 0x3
    event_lane = (event_info >> 4) & 0x3
    hardware_audit_html.value = (
        f"<pre style='margin:0;white-space:pre-wrap'>"
        f"Hardware Audit (Stage 15)\n"
        f"active_source={_source_label(source_name)}  control=0x{audit_control:08x}  status=0x{audit_status:08x}\n"
        f"event_valid={int(status.get('preview_event_valid', 0))}  event_active={int(status.get('preview_event_active', 0))}  overflow={int(status.get('preview_event_overflow', 0))}  threshold={int(status.get('preview_audit_event_threshold', 0) & 0xffff)}\n"
        f"start/first/done={int(status.get('preview_audit_start_count', 0))}/{int(status.get('preview_audit_first_count', 0))}/{int(status.get('preview_audit_done_count', 0))}  latency={int(status.get('preview_audit_start_to_first_latency', 0))} beats  capture_beats={int(status.get('preview_audit_capture_beats', 0))}\n"
        f"sample0 start/first/done={int(status.get('preview_audit_start_sample0', 0))}/{int(status.get('preview_audit_first_sample0', 0))}/{int(status.get('preview_audit_done_sample0', 0))}\n"
        f"valid_gap_count={int(status.get('preview_audit_valid_gap_count', 0))}  sample0_error_count={int(status.get('preview_audit_sample0_error_count', 0))}  nonmonotonic={int(status.get('preview_sample0_nonmonotonic', 0))}\n"
        f"event_sample0={int(status.get('preview_event_sample0', 0))}  event_max_code={int(status.get('preview_event_max_code', 0))}  event_source={_source_label({0:'rfdc',1:'internal_dds',2:'sample_index_ramp'}.get(event_source_code, event_source_code))} lane={event_lane}\n"
        f"event_rfdc_flags=0x{int(status.get('preview_event_rfdc_flags', 0)):08x}  event_dac_epoch={int(status.get('preview_event_dac_phase_epoch', 0))}\n"
        f"DAC audit: epoch_seen={int(status.get('dac_audit_phase_epoch_seen', 0))}  mode={dac_mode}  phase_acc=0x{int(status.get('dac_audit_ch0_phase_acc', 0)):08x}  phase_step=0x{int(status.get('dac_audit_ch0_phase_step', 0)):08x}  phase0=0x{int(status.get('dac_audit_ch0_phase0', 0)):08x}\n"
        f"Rule of thumb: internal_dds/ramp stable + RFDC unstable points below Jupyter, toward RFDC/analog/clock/adapter."
        f"</pre>"
    )

def apply_preview_audit(_=None):
    try:
        audit = core().configure_preview_audit(
            source=preview_audit_source.value,
            event_enable=bool(preview_event_enable.value),
            freeze_on_event=bool(preview_freeze_on_event.value),
            event_threshold=int(preview_event_threshold.value),
            clear=True,
        )
        update_hardware_audit_panel()
        set_status(f"Preview audit source set to {_source_label(audit['source'])}; event_enable={int(audit['event_enable'])}", kind='ok')
    except Exception as exc:
        set_status(f"Apply audit failed: {type(exc).__name__}: {exc}", kind='err')

def refresh_hardware_audit(_=None):
    try:
        update_hardware_audit_panel()
        set_status('Hardware audit status refreshed.', kind='ok')
    except Exception as exc:
        set_status(f"Refresh audit failed: {type(exc).__name__}: {exc}", kind='err')

def capture_hardware_event(_=None):
    try:
        if not bool(preview_event_enable.value):
            preview_event_enable.value = True
        core().configure_preview_audit(
            source=preview_audit_source.value,
            event_enable=True,
            freeze_on_event=bool(preview_freeze_on_event.value),
            event_threshold=int(preview_event_threshold.value),
            clear=True,
        )
        event = core().capture_preview_event(timeout=float(event_capture_timeout.value), n=256)
        state['last_hardware_event'] = event
        iq = np.asarray(event['iq'], dtype=np.int16)
        x = np.arange(iq.shape[0])
        with event_fig.batch_update():
            event_fig.data[0].x = x
            event_fig.data[0].y = iq[:, 0]
            event_fig.data[1].x = x
            event_fig.data[1].y = iq[:, 1]
            event_fig.layout.title = f"Preview large-event raw IQ buffer  max={int(event['max_code'])} sample0={int(event['sample0'])}"
        update_hardware_audit_panel()
        set_status(f"Captured preview event: max_code={int(event['max_code'])} sample0={int(event['sample0'])}", kind='warn')
    except Exception as exc:
        set_status(f"Capture event failed: {type(exc).__name__}: {exc}", kind='err')

def capture_rfdc_axis_raw_witness(_=None):
    try:
        ch = int(rfdc_raw_channel.value)
        beats = int(rfdc_raw_capture_beats.value)
        witness = core().capture_rfdc_axis_raw_witness(channel=ch, capture_beats=beats, timeout=float(timeout_s.value))
        state['last_rfdc_axis_raw_witness'] = witness
        iq = np.asarray(witness['decoded']['iq'], dtype=np.int16)
        x = np.arange(iq.shape[0], dtype=int)
        with rfdc_raw_fig.batch_update():
            rfdc_raw_fig.data[0].x = x
            rfdc_raw_fig.data[0].y = iq[:, 0] if iq.size else []
            rfdc_raw_fig.data[1].x = x
            rfdc_raw_fig.data[1].y = iq[:, 1] if iq.size else []
            rfdc_raw_fig.layout.title = f"RFDC AXIS raw witness  CH{ch}  beats={int(witness['beat_count'])} sample0={int(witness['sample0'])}"
        rfdc_raw_html.value = (
            f"<pre style='margin:0;white-space:pre-wrap'>"
            f"RFDC AXIS Raw Witness\n"
            f"source=RFDC output tap after mixer/decimator, before preview analysis\n"
            f"CH{ch} beats={int(witness['beat_count'])}/{beats} words={int(witness['word_count'])} sample0={int(witness['sample0'])}\n"
            f"rfdc_flags=0x{int(witness['rfdc_flags']):08x} valid_mask=0x{int(witness['valid_mask']):04x}\n"
            f"waveform_source=rfdc_axis_raw_witness_buffer virtual_waveform=False"
            f"</pre>"
        )
        set_status(f"Captured RFDC AXIS raw witness: CH{ch} beats={int(witness['beat_count'])} sample0={int(witness['sample0'])}", kind='ok')
    except Exception as exc:
        set_status(f"Capture RFDC raw witness failed: {type(exc).__name__}: {exc}", kind='err')

def update_status(status, analysis, rates, elapsed_s, display_state):
    fps = 1.0 / elapsed_s if elapsed_s > 0 else 0.0
    state['last_fps'] = fps
    wf_len = len(state['waterfall'])
    lines = [
        f"{observation_label()}  center={float(center_mhz.value):.1f} MHz  BW={float(bandwidth_mhz.value):.0f} MHz  FPS={fps:.2f}",
        format_rates(rates),
        f"UDP_DRY_RUN={int(rates['udp_dry_run'])}  QSFP_PRESENT={int(rates.get('qsfp_module_present', False))}  QSFP_LINK_UP={int(rates['qsfp_link_up'])}  CMAC_READY={int(rates.get('cmac_live_ready', False))}  PPS_COUNT={int(status.get('pps_count', 0))}  PPS_RECENT={int(status.get('pps_recent', 0))}  sample0={analysis['sample0']}  nsamp={analysis['count']}  waterfall=CH{int(waterfall_channel.value)} {wf_len}/{int(waterfall_depth.value)}",
    ]
    prof = state.get('last_profile', {})
    if prof:
        lines.append(
            f"timing ms: capture={prof.get('capture_ms', 0.0):.1f} analysis={prof.get('analysis_ms', 0.0):.1f} figures={prof.get('figures_ms', 0.0):.1f} total={prof.get('total_ms', 0.0):.1f}"
        )
    for ch in analysis['inputs']:
        pk = analysis['peaks'][ch]
        ds = display_state.get(ch, {})
        reject = ds.get('reject_reason', '')
        reject_txt = f"  display_hold_by_stabilizer={reject}" if reject else ""
        lines.append(
            f"CH{ch}: {CHANNEL_LABELS[ch]}  RF peak={float(ds.get('display_peak_mhz', pk['rf_peak_mhz'])):.4f} MHz  "
            f"peak={float(ds.get('display_peak_dbfs', pk.get('peak_dbfs', -240.0))):.1f} dBFS  "
            f"rms={float(ds.get('display_rms_dbfs', pk.get('rms_dbfs', -240.0))):.1f} dBFS  "
            f"raw_peak={float(pk.get('peak_dbfs', -240.0)):.1f} dBFS  "
            f"fit_phase={float(pk.get('coherent_phase_deg', pk.get('phase_deg', 0.0))):+.1f} deg  SNR={float(pk['snr_db']):.1f} dB  max={float(pk['max_abs_code']):.0f}{reject_txt}"
        )
    status_kind = 'warn' if any(str(item.get('reject_reason', '')) for item in display_state.values()) else 'ok'
    set_status('\n'.join(lines), kind=status_kind)
    update_phase_provenance_panel(analysis.get('phase_provenance'), analysis.get('sample0_aligned_view'))
    update_hardware_audit_panel(status)
    ch0 = display_state.get(0, {})
    advanced_html.value = (
        f"<pre style='margin:0;white-space:pre-wrap'>"
        f"CORE_VERSION=0x{int(status['core_version']):08x}  PPS_COUNT={int(status.get('pps_count', 0))}  PPS_RECENT={int(status.get('pps_recent', 0))}  PPS_IN={int(status.get('pps_input_high', 0))}\n"
        f"RFDC_SYSREF_LOCK={'PASS' if state.get('last_cfg', {}).get('nco', {}).get('configured', False) else 'UNKNOWN'}  paired_valid={int(status.get('tx_paired_witness_valid', 0))}  paired_preview_done={int(status.get('tx_paired_preview_done', 0))}  sample0_delta={int(status.get('tx_paired_sample0_delta', 0))}\n"
        f"LMK_LOCK={'PASS' if state.get('last_cfg', {}).get('clock', {}).get('configured', False) else 'FAIL/UNKNOWN'}  pll1={state.get('last_cfg', {}).get('clock', {}).get('pll1_lock', '?')}  pll2={state.get('last_cfg', {}).get('clock', {}).get('pll2_lock', '?')}  profile={state.get('last_cfg', {}).get('clock', {}).get('profile_id', 'unknown')}\n"
        f"RFDC_MTS_CAPABILITY={state.get('last_cfg', {}).get('rfdc_driver', {}).get('classification', 'unknown')}  shim_ready={state.get('last_cfg', {}).get('rfdc_driver', {}).get('cffi_mts_shim_ready', False)}  direct_api={state.get('last_cfg', {}).get('rfdc_driver', {}).get('direct_python_mts_api', False)}\n"
        f"MTS_ADC_LATENCY={state.get('last_cfg', {}).get('nco', {}).get('mts', {}).get('adc_config', {}).get('latency', [])}  MTS_DAC_LATENCY={state.get('last_cfg', {}).get('nco', {}).get('mts', {}).get('dac_config', {}).get('latency', [])}\n"
        f"PREVIEW_PAYLOAD_GATE={'READY' if state.get('last_cfg', {}).get('stage18_mts_locked', False) else 'BLOCKED_PENDING_STAGE18_GATE'}  UDP_DRY_RUN={int(rates['udp_dry_run'])}  QSFP_PRESENT={int(rates.get('qsfp_module_present', False))}  QSFP_LINK_UP={int(rates['qsfp_link_up'])}  CMAC_READY={int(rates.get('cmac_live_ready', False))}\n"
        f"SCIENCE_MODE={rates.get('science_output_mode', 'UNKNOWN')}  SCIENCE_BW={rates.get('science_bandwidth_mhz', 0)} MHz  SCIENCE_FS={rates.get('science_sample_rate_hz', 0.0)/1e6:.2f} MS/s  SCIENCE_PAYLOAD={rates.get('science_payload_rate_mbps', 0.0):.0f} Mb/s  BLOCK={','.join(rates.get('science_block_reasons', [])) or 'none'}\n"
        f"preview_sample_rate={analysis['sample_rate_hz']/1e6:.2f} MS/s  axis_beat={analysis['axis_beat_rate_hz']/1e6:.2f} MHz\n"
        f"input_source={analysis.get('input_source_mode', 'unknown')}  expected_signal={analysis.get('expected_signal_hz', 0.0)/1e6:.4f} MHz  dac_signal={analysis.get('dac_signal_hz', 0.0)/1e6:.4f} MHz\n"
        f"phase_lock={analysis.get('phase_lock', 'unknown')}  expected_baseband={analysis['expected_baseband_hz']/1e6:+.4f} MHz  nfft={analysis['nfft']}  oversample={analysis['oversample']:.2f}\n"
        f"raw_baseband_CH0={analysis['peaks'].get(0, {}).get('raw_baseband_mhz', 0.0):+.4f} MHz  mixer_sign={analysis['peaks'].get(0, {}).get('mixer_sign', 0)}\n"
        f"CH0 raw_peak={ch0.get('raw_peak_dbfs', 0.0):.2f} dBFS  smoothed_peak={ch0.get('display_peak_dbfs', 0.0):.2f} dBFS  raw_rms={ch0.get('raw_rms_dbfs', 0.0):.2f} dBFS  smoothed_rms={ch0.get('display_rms_dbfs', 0.0):.2f} dBFS\n"
        f"CH0 valid_frame={ch0.get('valid_frame', True)}  reject_reason={ch0.get('reject_reason', '')}\n"
        f"RFDC valid=0x{int(status['rfdc_current_valid_mask']):04x}  tx_status=0x{int(status.get('tx_status', 0)):08x}\n"
        f"</pre>"
    )

def update_waterfall_plot(selected_update):
    if not bool(waterfall_live.value):
        return False
    center = float(center_mhz.value)
    bw = float(bandwidth_mhz.value)
    xmin = center - bw / 2.0
    xmax = center + bw / 2.0
    now = time.monotonic()
    interval = 1.0 / max(float(waterfall_update_hz.value), 0.1)
    appended = False
    if selected_update is not None and selected_update.get('accepted', False):
        if now - float(state.get('last_waterfall_update', 0.0)) >= interval:
            row = selected_update['raw_power_dbfs'] if bool(raw_spectrum.value) else selected_update['display_power_dbfs']
            x_reduced, z_reduced = reduce_spectrum_bins(
                selected_update['rf_mhz'],
                row,
                int(waterfall_bins.value),
                xmin,
                xmax,
                float(spectrum_floor_db.value),
            )
            state['waterfall'].append({
                'x': x_reduced,
                'z': z_reduced,
                't': now,
            })
            state['last_waterfall_update'] = now
            appended = True
    if not appended:
        return False
    with waterfall_fig.batch_update():
        rows = list(state['waterfall'])
        waterfall_fig.data[0].x = rows[-1]['x']
        waterfall_fig.data[0].y = np.arange(-len(rows) + 1, 1, dtype=int)
        waterfall_fig.data[0].z = np.vstack([item['z'] for item in rows])
        waterfall_fig.data[0].zmin = float(spectrum_floor_db.value)
        waterfall_fig.data[0].zmax = float(spectrum_top_db.value)
        waterfall_fig.layout.xaxis.range = [xmin, xmax]
        waterfall_fig.layout.yaxis.range = [-max(int(waterfall_depth.value) - 1, 1), 0]
        waterfall_fig.layout.title = f"Waterfall  CH{int(waterfall_channel.value)}  history={len(state['waterfall'])}/{int(waterfall_depth.value)}  {float(waterfall_update_hz.value):.1f} Hz"
    return True

def update_figures(analysis):
    if state['waterfall_key'] != waterfall_signature():
        reset_stability_history(clear_plot=True)
    center = float(center_mhz.value)
    bw = float(bandwidth_mhz.value)
    xmin = center - bw / 2.0
    xmax = center + bw / 2.0
    y = int(scope_y_codes.value)
    mask = visible_mask()
    waterfall_ch = int(waterfall_channel.value)
    layout_key = (
        round(center, 6), round(bw, 6), round(float(time_window_us.value), 6), y, str(scope_waveform_mode.value),
        int(spectrum_floor_db.value), int(spectrum_top_db.value), int(scope_display_points.value),
        int(spectrum_display_points.value), int(waterfall_bins.value), bool(baseband_live.value), bool(waterfall_live.value),
        bool(show_measured_aligned_scope.value), state.get('alignment_anchor_deg') is not None,
    )
    update_layout = layout_key != state.get('last_layout_key')
    state['last_layout_key'] = layout_key
    display_state = {}
    selected_update = None
    for ch in range(8):
        wants_channel = bool(mask & (1 << ch)) or (bool(waterfall_live.value) and ch == waterfall_ch)
        if ch in analysis['spectrum'] and wants_channel:
            sp = analysis['spectrum'][ch]
            pk = analysis['peaks'][ch]
            ds = state['stabilizer'].update_channel(
                ch,
                sp,
                pk,
                smoothing_enabled=bool(smoothing_enabled.value),
                alpha=float(smoothing_alpha.value),
            )
            display_state[ch] = ds
            if ch == waterfall_ch:
                selected_update = ds
    with scope_fig.batch_update():
        scope_mode = str(scope_waveform_mode.value)
        for ch in range(8):
            i_idx = 2 * ch
            q_idx = i_idx + 1
            if not (mask & (1 << ch)) or ch not in analysis['baseband_scope']:
                scope_fig.data[i_idx].visible = False
                scope_fig.data[q_idx].visible = False
                continue
            tr = analysis['baseband_scope'][ch]
            raw_i = np.asarray(tr.get('raw_waveform', []), dtype=float)
            raw_q = np.asarray(tr.get('raw_q_waveform', np.zeros_like(raw_i)), dtype=float)
            raw_mag = np.asarray(tr.get('raw_magnitude_waveform', np.sqrt(raw_i * raw_i + raw_q * raw_q)), dtype=float)
            rf_equiv = np.asarray(tr.get('rf_equivalent_waveform', np.zeros_like(raw_i)), dtype=float)
            points = int(scope_display_points.value)
            if bool(fast_mode.value):
                points = min(points, 512)
            if scope_mode == 'raw_mag':
                x_scope, y_scope = reduce_uniform(tr['time_us'], raw_mag, points)
                scope_fig.data[i_idx].name = f'CH{ch} |IQ|'
                scope_fig.data[i_idx].x = x_scope
                scope_fig.data[i_idx].y = y_scope
                scope_fig.data[i_idx].visible = True
                scope_fig.data[q_idx].visible = False
            elif scope_mode == 'rf_equiv':
                x_scope, y_scope = reduce_uniform(tr.get('rf_equivalent_time_us', tr['time_us']), rf_equiv, points)
                scope_fig.data[i_idx].name = f'CH{ch} RF equiv'
                scope_fig.data[i_idx].x = x_scope
                scope_fig.data[i_idx].y = y_scope
                scope_fig.data[i_idx].visible = True
                scope_fig.data[q_idx].visible = False
            elif scope_mode == 'raw_q':
                x_scope, y_scope = reduce_uniform(tr['time_us'], raw_q, points)
                scope_fig.data[i_idx].name = f'CH{ch} Q'
                scope_fig.data[i_idx].x = x_scope
                scope_fig.data[i_idx].y = y_scope
                scope_fig.data[i_idx].visible = True
                scope_fig.data[q_idx].visible = False
            elif scope_mode == 'iq_overlay':
                x_i, y_i = reduce_uniform(tr['time_us'], raw_i, points)
                x_q, y_q = reduce_uniform(tr['time_us'], raw_q, points)
                scope_fig.data[i_idx].name = f'CH{ch} I'
                scope_fig.data[i_idx].x = x_i
                scope_fig.data[i_idx].y = y_i
                scope_fig.data[i_idx].visible = True
                scope_fig.data[q_idx].name = f'CH{ch} Q'
                scope_fig.data[q_idx].x = x_q
                scope_fig.data[q_idx].y = y_q
                scope_fig.data[q_idx].visible = True
            else:
                x_scope, y_scope = reduce_uniform(tr['time_us'], raw_i, points)
                scope_fig.data[i_idx].name = f'CH{ch} I'
                scope_fig.data[i_idx].x = x_scope
                scope_fig.data[i_idx].y = y_scope
                scope_fig.data[i_idx].visible = True
                scope_fig.data[q_idx].visible = False
        if update_layout:
            title_by_mode = {'raw_i': 'RFDC preview I', 'raw_q': 'RFDC preview Q', 'iq_overlay': 'RFDC preview I/Q', 'raw_mag': 'RFDC preview |IQ|', 'rf_equiv': 'RF等效(I/Q回算) derived_from_real_iq=True raw_rf=False'}
            scope_fig.layout.xaxis.range = [0, float(time_window_us.value)]
            scope_fig.layout.xaxis.title = 'preview time (us)'
            scope_fig.layout.yaxis.range = [0, y] if scope_mode == 'raw_mag' else [-y, y]
            scope_fig.layout.title = f"{title_by_mode.get(scope_mode, scope_mode)}  {observation_label()}  window={float(time_window_us.value):.2f} us"
    aligned_view = analysis.get('sample0_aligned_view')
    aligned_channels = {} if aligned_view is None else aligned_view.get('channels', {})
    history_by_channel = state.setdefault('aligned_history_by_channel', {ch: deque(maxlen=240) for ch in range(8)})
    visible_aligned_channels = []
    if aligned_view is not None:
        state['last_aligned_view'] = aligned_view
        state['aligned_history'].clear()
        for ch in range(8):
            aligned_item = aligned_channels.get(ch) or aligned_channels.get(str(ch))
            if aligned_item is not None and (mask & (1 << ch)):
                visible_aligned_channels.append(ch)
                history_by_channel.setdefault(ch, deque(maxlen=240)).append(float(aligned_item['phase_error_deg']))
        if 0 in visible_aligned_channels:
            state['aligned_history'].extend(history_by_channel.get(0, []))
    with aligned_scope_fig.batch_update():
        points = min(int(scope_display_points.value), 512 if bool(fast_mode.value) else int(scope_display_points.value))
        for ch in range(8):
            i_idx = 2 * ch
            q_idx = i_idx + 1
            aligned_item = aligned_channels.get(ch) or aligned_channels.get(str(ch))
            if aligned_item is not None and (mask & (1 << ch)) and bool(show_measured_aligned_scope.value):
                raw_i = np.asarray(aligned_item.get('preview_waveform_i', []), dtype=float)
                raw_q = np.asarray(aligned_item.get('preview_waveform_q', np.zeros_like(raw_i)), dtype=float)
                raw_mag = np.asarray(aligned_item.get('preview_waveform_mag', np.sqrt(raw_i * raw_i + raw_q * raw_q)), dtype=float)
                rf_equiv = np.asarray(aligned_item.get('rf_equivalent_waveform', np.zeros_like(raw_i)), dtype=float)
                x_raw = aligned_item.get('preview_time_us', aligned_item.get('time_us', []))
                if scope_mode == 'raw_mag':
                    x_i, y_i = reduce_uniform(x_raw, raw_mag, points)
                    aligned_scope_fig.data[i_idx].name = f'CH{ch} |IQ|'
                    aligned_scope_fig.data[i_idx].x = x_i
                    aligned_scope_fig.data[i_idx].y = y_i
                    aligned_scope_fig.data[i_idx].visible = True
                    aligned_scope_fig.data[q_idx].visible = False
                elif scope_mode == 'rf_equiv':
                    x_i, y_i = reduce_uniform(aligned_item.get('rf_equivalent_time_us', x_raw), rf_equiv, points)
                    aligned_scope_fig.data[i_idx].name = f'CH{ch} RF equiv'
                    aligned_scope_fig.data[i_idx].x = x_i
                    aligned_scope_fig.data[i_idx].y = y_i
                    aligned_scope_fig.data[i_idx].visible = True
                    aligned_scope_fig.data[q_idx].visible = False
                elif scope_mode == 'raw_q':
                    x_i, y_i = reduce_uniform(x_raw, raw_q, points)
                    aligned_scope_fig.data[i_idx].name = f'CH{ch} Q'
                    aligned_scope_fig.data[i_idx].x = x_i
                    aligned_scope_fig.data[i_idx].y = y_i
                    aligned_scope_fig.data[i_idx].visible = True
                    aligned_scope_fig.data[q_idx].visible = False
                elif scope_mode == 'iq_overlay':
                    x_i, y_i = reduce_uniform(x_raw, raw_i, points)
                    x_q, y_q = reduce_uniform(x_raw, raw_q, points)
                    aligned_scope_fig.data[i_idx].name = f'CH{ch} I'
                    aligned_scope_fig.data[i_idx].x = x_i
                    aligned_scope_fig.data[i_idx].y = y_i
                    aligned_scope_fig.data[i_idx].visible = True
                    aligned_scope_fig.data[q_idx].name = f'CH{ch} Q'
                    aligned_scope_fig.data[q_idx].x = x_q
                    aligned_scope_fig.data[q_idx].y = y_q
                    aligned_scope_fig.data[q_idx].visible = True
                else:
                    x_i, y_i = reduce_uniform(x_raw, raw_i, points)
                    aligned_scope_fig.data[i_idx].name = f'CH{ch} I'
                    aligned_scope_fig.data[i_idx].x = x_i
                    aligned_scope_fig.data[i_idx].y = y_i
                    aligned_scope_fig.data[i_idx].visible = True
                    aligned_scope_fig.data[q_idx].visible = False
            else:
                aligned_scope_fig.data[i_idx].visible = False
                aligned_scope_fig.data[q_idx].visible = False
        if update_layout:
            aligned_scope_fig.layout.xaxis.range = [0, float(time_window_us.value)]
            aligned_scope_fig.layout.yaxis.range = [-y, y]
            anchor_txt = 'anchored' if state.get('alignment_anchor_deg') is not None else 'unanchored'
            ch_txt = ','.join(f'CH{ch}' for ch in visible_aligned_channels) if visible_aligned_channels else 'no channels'
            aligned_scope_fig.layout.title = f"Sample0-indexed raw preview scope  {ch_txt}  {anchor_txt}"
    with aligned_phase_fig.batch_update():
        latest = []
        for ch in range(8):
            hist = list(history_by_channel.get(ch, []))
            aligned_phase_fig.data[ch].x = np.arange(-len(hist) + 1, 1, dtype=int) if hist else []
            aligned_phase_fig.data[ch].y = hist
            aligned_phase_fig.data[ch].visible = bool(hist) and bool(mask & (1 << ch))
            if hist and (mask & (1 << ch)):
                latest.append(f"CH{ch}={hist[-1]:+.2f}")
        latest_txt = '  '.join(latest[:4]) + (' ...' if len(latest) > 4 else '')
        aligned_phase_fig.layout.title = f"Sample0-aligned phase residual  {latest_txt}"
    if bool(baseband_live.value):
        with baseband_fig.batch_update():
            for ch in range(8):
                if ch in analysis['baseband_scope'] and (mask & (1 << ch)):
                    tr = analysis['baseband_scope'][ch]
                    x_base, y_base = reduce_uniform(
                        tr['time_us'],
                        tr['raw_waveform'],
                        min(int(scope_display_points.value), 512 if bool(fast_mode.value) else int(scope_display_points.value)),
                    )
                    baseband_fig.data[ch].x = x_base
                    baseband_fig.data[ch].y = y_base
                    if baseband_fig.data[ch].visible is not True:
                        baseband_fig.data[ch].visible = True
                elif baseband_fig.data[ch].visible is not False:
                    baseband_fig.data[ch].visible = False
            if update_layout:
                baseband_fig.layout.xaxis.range = [0, float(time_window_us.value)]
                baseband_fig.layout.yaxis.range = [-y, y]
                baseband_fig.layout.title = f"Baseband debug scope  sample0={analysis['sample0']}"
    with spectrum_fig.batch_update():
        for ch in range(8):
            if ch in display_state and (mask & (1 << ch)):
                ds = display_state[ch]
                ydata = ds['raw_power_dbfs'] if bool(raw_spectrum.value) else ds['display_power_dbfs']
                points = int(spectrum_display_points.value)
                if bool(fast_mode.value):
                    points = min(points, 384)
                x_spec, y_spec = reduce_spectrum_peak(
                    ds['rf_mhz'],
                    ydata,
                    points,
                    xmin,
                    xmax,
                )
                spectrum_fig.data[ch].x = x_spec
                spectrum_fig.data[ch].y = y_spec
                if spectrum_fig.data[ch].visible is not True:
                    spectrum_fig.data[ch].visible = True
            elif spectrum_fig.data[ch].visible is not False:
                spectrum_fig.data[ch].visible = False
        if update_layout:
            spectrum_fig.layout.xaxis.range = [xmin, xmax]
            spectrum_fig.layout.yaxis.range = [int(spectrum_floor_db.value), int(spectrum_top_db.value)]
            mode = 'raw' if bool(raw_spectrum.value) else ('smoothed' if bool(smoothing_enabled.value) else 'single-frame')
            spectrum_fig.layout.title = f"Spectrum ({mode})  center={center:.1f} MHz  BW={bw:.0f} MHz  {int(spectrum_display_points.value)} pts"
    update_waterfall_plot(selected_update)
    state['last_display'] = display_state
    return display_state

def capture_once(_=None):
    try:
        c = core()
        total_t0 = time.monotonic()
        t0 = time.monotonic()
        if bool(freeze_frame.value) and state.get('frozen_preview') is not None:
            preview = state['frozen_preview']
        else:
            preview = c.capture_preview_fast(n=capture_count(), input_mask=capture_input_mask(), timeout=float(timeout_s.value))
            state['last_preview'] = preview
            if bool(freeze_frame.value):
                state['frozen_preview'] = preview
        t1 = time.monotonic()
        analysis = c.compute_observation_view(
            preview,
            observe_center_hz=float(center_mhz.value) * 1_000_000.0,
            view_bw_hz=float(bandwidth_mhz.value) * 1_000_000.0,
            dac_signal_hz=dac_signal_hz(),
            expected_signal_hz=expected_signal_hz(),
            time_window_us=float(time_window_us.value),
            oversample=float(oversample.value),
            phase_ref_input=0,
            stabilize_phase=False,
            display_phase_deg=0.0,
            phase_deg_by_channel=phase_values(),
            input_source_mode=input_source(),
        )
        analysis['phase_provenance'] = c.compute_phase_provenance(
            preview,
            observe_center_hz=float(center_mhz.value) * 1_000_000.0,
            view_bw_hz=float(bandwidth_mhz.value) * 1_000_000.0,
            dac_signal_hz=dac_signal_hz(),
            expected_signal_hz=expected_signal_hz(),
            configured_phase_deg=0.0,
            display_phase_deg=0.0,
            phase_deg_by_channel=phase_values(),
            phase_ref_input=0,
            oversample=8.0,
            input_source_mode=input_source(),
        )
        aligned_view = c.compute_sample0_aligned_phase_view(
            preview,
            observe_center_hz=float(center_mhz.value) * 1_000_000.0,
            dac_signal_hz=dac_signal_hz(),
            expected_signal_hz=expected_signal_hz(),
            configured_phase_deg=0.0,
            phase_deg_by_channel=phase_values(),
            alignment_anchor_deg=state.get('alignment_anchor_deg'),
            phase_ref_input=0,
            time_window_us=float(time_window_us.value),
            display_points=min(int(scope_display_points.value), 512 if bool(fast_mode.value) else int(scope_display_points.value)),
            fft_oversample=8.0,
            input_source_mode=input_source(),
        )
        ch0_aligned = aligned_view.get('channels', {}).get(0) or aligned_view.get('channels', {}).get('0')
        if ch0_aligned is not None and bool(state.get('auto_anchor_pending', False)):
            state['alignment_anchor_deg'] = float(ch0_aligned['anchor_candidate_deg'])
            state['auto_anchor_pending'] = False
            state['aligned_history'].clear()
            for hist in state.get('aligned_history_by_channel', {}).values():
                hist.clear()
            aligned_view = c.compute_sample0_aligned_phase_view(
                preview,
                observe_center_hz=float(center_mhz.value) * 1_000_000.0,
                dac_signal_hz=dac_signal_hz(),
                expected_signal_hz=expected_signal_hz(),
                configured_phase_deg=0.0,
                phase_deg_by_channel=phase_values(),
                alignment_anchor_deg=state.get('alignment_anchor_deg'),
                phase_ref_input=0,
                time_window_us=float(time_window_us.value),
                display_points=min(int(scope_display_points.value), 512 if bool(fast_mode.value) else int(scope_display_points.value)),
                fft_oversample=8.0,
                input_source_mode=input_source(),
            )
        analysis['sample0_aligned_view'] = aligned_view
        t2 = time.monotonic()
        now = time.monotonic()
        status_interval = 1.0 / max(float(status_update_hz.value), 0.1)
        status_due = (
            state.get('last_rates') is None
            or now - float(state.get('last_status_update', 0.0)) >= status_interval
            or not state.get('live', False)
            or bool(freeze_frame.value)
        )
        rates = get_rates(c, force=status_due)
        t3 = time.monotonic()
        display_state = update_figures(analysis)
        t4 = time.monotonic()
        elapsed = t4 - total_t0
        state['last_capture_elapsed_s'] = elapsed
        state['last_profile'] = {
            'capture_ms': (t1 - t0) * 1000.0,
            'analysis_ms': (t2 - t1) * 1000.0,
            'rates_ms': (t3 - t2) * 1000.0,
            'figures_ms': (t4 - t3) * 1000.0,
            'total_ms': elapsed * 1000.0,
        }
        if status_due:
            update_status(rates['status'], analysis, rates, elapsed, display_state)
            state['last_status_update'] = time.monotonic()
    except Exception as exc:
        state['last_error'] = exc
        set_status(f"Capture failed: {type(exc).__name__}: {exc}", kind='err')

async def live_loop():
    while state['live']:
        t0 = time.monotonic()
        capture_once()
        elapsed = time.monotonic() - t0
        target_sleep = 1.0 / max(float(refresh_hz.value), 0.1) - elapsed
        min_sleep = max(0.0, float(min_ui_sleep_ms.value) / 1000.0)
        await asyncio.sleep(max(min_sleep, target_sleep))

def start_live(_=None):
    if state['live']:
        return
    state['live'] = True
    state['task'] = asyncio.create_task(live_loop())
    set_status('Live started', kind='ok')

def stop_live(_=None):
    state['live'] = False
    set_status('Live stopped', kind='warn')

load_btn.on_click(load_overlay)
init_btn.on_click(init_instrument)
apply_btn.on_click(apply_instrument)
capture_btn.on_click(capture_once)
start_btn.on_click(start_live)
stop_btn.on_click(stop_live)
reset_history_btn.on_click(reset_stability_history)
set_anchor_btn.on_click(set_alignment_anchor)
apply_audit_btn.on_click(apply_preview_audit)
refresh_audit_btn.on_click(refresh_hardware_audit)
capture_event_btn.on_click(capture_hardware_event)
capture_rfdc_raw_btn.on_click(capture_rfdc_axis_raw_witness)
sync_diag_btn.on_click(diagnose_sync)
freeze_frame.observe(on_freeze_change, names='value')

def apply_global_phase_to_channels(_=None):
    for widget in phase_ch:
        widget.value = float(phase_deg.value)
    if state.get('core') is not None:
        on_phase_change({'new': phase_deg.value})

apply_global_phase_btn.on_click(apply_global_phase_to_channels)
phase_deg.observe(lambda change: None, names='value')
for _phase_widget in phase_ch:
    _phase_widget.observe(on_phase_change, names='value')
input_source_mode.observe(lambda change: reset_stability_history(clear_plot=True), names='value')
expected_signal_mhz.observe(lambda change: reset_stability_history(clear_plot=True), names='value')
bandwidth_mhz.observe(lambda change: reset_stability_history(clear_plot=True), names='value')
science_output_mode.observe(lambda change: reset_stability_history(clear_plot=False), names='value')
visible_channels.observe(lambda change: update_channel_visibility(), names='value')
waterfall_channel.observe(on_waterfall_setting_change, names='value')
waterfall_depth.observe(on_waterfall_setting_change, names='value')

control_bar = W.HBox([load_btn, init_btn, apply_btn, capture_btn, start_btn, stop_btn, reset_history_btn, freeze_frame, set_anchor_btn])
science_box = W.VBox([input_source_mode, expected_signal_mhz, signal_mhz, center_mhz, bandwidth_mhz, science_output_mode, W.HBox([force_tx_dry_run, cmac_enable]), time_window_us, oversample, clock_ref_mode, sync_mode_select, require_mts_lock, force_clock_reconfigure])
phase_grid = W.GridBox(phase_ch, layout=W.Layout(grid_template_columns='repeat(2, 310px)', grid_gap='4px 8px'))
display_box = W.VBox([amplitude_percent, W.HBox([phase_deg, apply_global_phase_btn]), phase_grid, scope_waveform_mode, scope_y_codes, spectrum_floor_db, spectrum_top_db, refresh_hz, visible_channels, smoothing_enabled, smoothing_alpha, waterfall_channel, waterfall_depth, auto_anchor_after_apply, show_measured_aligned_scope])
performance_box = W.VBox([fast_mode, baseband_live, waterfall_live, scope_display_points, spectrum_display_points, waterfall_bins, waterfall_update_hz, status_update_hz, min_ui_sleep_ms])
sync_box = W.VBox([W.HBox([sync_diag_btn]), sync_html])
hardware_box = W.VBox([sync_box, W.HBox([preview_audit_source, apply_audit_btn, refresh_audit_btn, capture_event_btn]), preview_event_enable, preview_freeze_on_event, preview_event_threshold, event_capture_timeout, hardware_audit_html, event_fig, W.HBox([rfdc_raw_channel, rfdc_raw_capture_beats, capture_rfdc_raw_btn]), rfdc_raw_html, rfdc_raw_fig])
advanced = W.Accordion(children=[W.VBox([download_bitstream, timeout_s, raw_spectrum, performance_box, phase_html, phase_fig, baseband_fig, advanced_html]), hardware_box])
advanced.set_title(0, '高级工程状态')
advanced.set_title(1, 'Hardware Audit')
header = W.HTML("<h2 style='margin:0 0 8px 0'>T510 F-engine Astronomer RF Observation Console</h2>")
body = W.VBox([
    header,
    control_bar,
    W.HBox([science_box, display_box], layout=W.Layout(gap='24px')),
    status_html,
    W.HBox([scope_fig, spectrum_fig]),
    W.HBox([aligned_scope_fig, aligned_phase_fig]),
    waterfall_fig,
    advanced,
], layout=W.Layout(gap='10px'))
set_status(f"Ready. Project={PROJECT_ROOT}\nOpen overlay: {BITFILE}", kind='info')
display(body)


### RFDC-to-UDP Coherence Witness

Advanced Stage 18 check: compare the current MTS/SYSREF-locked RFDC preview frame with the T510 packet payload captured just before UDP frame building.

In [ ]:
coherence_mode = W.Dropdown(options=[('SPEC payload', 'spec'), ('TIME payload', 'time')], value='spec', description='Payload', style={'description_width': '90px'}, layout=W.Layout(width='260px'))
coherence_capture_words = W.IntSlider(value=128, min=32, max=128, step=16, description='Words', style={'description_width': '90px'}, layout=W.Layout(width='300px'))
coherence_btn = W.Button(description='Capture coherence witness', button_style='warning')
coherence_out = W.Output()

def capture_coherence_witness(_=None):
    with coherence_out:
        coherence_out.clear_output(wait=True)
        c = core()
        signal_hz = expected_signal_hz()
        dac_hz = dac_signal_hz()
        center_hz = float(center_mhz.value) * 1_000_000.0
        preview = c.capture_preview_fast(n=capture_count(), input_mask=0x01, timeout=float(timeout_s.value))
        preview_view = T510FEngine.compute_sample0_aligned_phase_view(
            preview,
            observe_center_hz=center_hz,
            dac_signal_hz=dac_hz,
            expected_signal_hz=signal_hz,
            configured_phase_deg=float(phase_deg.value),
            alignment_anchor_deg=0.0,
            phase_ref_input=0,
            time_window_us=float(time_window_us.value),
            display_points=128,
            fft_oversample=4.0,
            input_source_mode=input_source(),
        )
        preview_ch0 = preview_view['channels'].get(0) or preview_view['channels'].get('0')
        witness = c.capture_tx_payload_witness(coherence_mode.value, timeout=float(timeout_s.value), capture_words=int(coherence_capture_words.value))
        if coherence_mode.value == 'spec':
            decoded = T510FEngine.decode_spec_payload_iq(witness, channel=0)
        else:
            decoded = T510FEngine.decode_time_payload_iq(witness, channel=0)
        payload_metrics = T510FEngine.compute_payload_phase_metrics(
            decoded,
            sample_rate_hz=245_760_000.0,
            observe_center_hz=center_hz,
            dac_signal_hz=dac_hz,
            expected_signal_hz=signal_hz,
            configured_phase_deg=float(phase_deg.value),
        )
        summary = {
            'mode': coherence_mode.value,
            'preview_sample0': int(preview['sample0']),
            'payload_sample0': int(witness['header'].sample0),
            'sample0_delta': int(witness['header'].sample0) - int(preview['sample0']),
            'preview_phase_error_deg': float(preview_ch0['phase_error_deg']),
            'payload_phase_error_deg': float(payload_metrics['phase_error_deg']),
            'preview_amplitude_code': float(preview_ch0['amplitude_code']),
            'payload_amplitude_code': float(payload_metrics['amplitude_code']),
            'payload_decoded_count': int(decoded['decoded_count']),
            'header': witness['header_dict'],
            'metadata': witness['metadata'],
        }
        state['last_coherence_witness'] = summary
        print(json.dumps(summary, indent=2, sort_keys=True))

coherence_btn.on_click(capture_coherence_witness)
display(W.VBox([W.HBox([coherence_mode, coherence_capture_words, coherence_btn]), coherence_out]))
